#1.7写一个调用脚本

In [2]:
import os
from turtledemo.penrose import start

from openai import OpenAI
from dotenv import load_dotenv
from pyexpat.errors import messages

load_dotenv()  #读取.env文件

client = OpenAI(

    api_key=os.getenv("DEEPSEEK_API_KEY"),
    base_url="https://api.deepseek.com"
)

response = client.chat.completions.create(
    model="deepseek-v4-flash",
    messages=[
        {
            "role": "user",
            "content": "用一句话介绍你自己"
        }
    ]
)

print(response.choices[0].message.content)

我是DeepSeek，由深度求索打造的AI助手，乐于为你解答问题、提供帮助！


#3.2看透响应对象

In [4]:
#from langchain.chat_models import init_chat_model
from openai import OpenAI
from dotenv import load_dotenv
import os

load_dotenv()

client = OpenAI(
    api_key=os.getenv("DEEPSEEK_API_KEY"),
    base_url="https://api.deepseek.com"
)

response = client.chat.completions.create(
    model="deepseek-v4-flash",
    messages=[{
        "role": "user",
        "content": "1+1=?"
    }]
)

print("回答:", response.choices[0].message.content)

print("角色:", response.choices[0].message.role)

print("本次花费tokens:", response.usage.total_tokens)

print("提示tokens:", response.usage.prompt_tokens)

print("生成tokens:", response.usage.completion_tokens)
#看懂usage,算成本算延迟都靠它

回答: 2
角色: assistant
本次花费tokens: 105
提示tokens: 87
生成tokens: 18


#3.3temperature对比实验

In [ ]:
from openai import OpenAI
from dotenv import load_dotenv
import os

load_dotenv()

client = OpenAI(api_key=os.getenv("DEEPSEEK_API_KEY"), base_url="https://api.deepseek.com")

question = "给我讲一个关于程序员的笑话"

for temp in [0, 0.7, 1.5]:
    resp = client.chat.completions.create(
        model="deepseek-v4-flash",
        messages=[
            {"role": "system",
             "content": "你是一个友好的助手"},
            {"role": "user",
             "content": "我叫李雷"},
            {"role": "assistant",
             "content": "你好李雷！很高兴认识你。"},
            {"role": "user",
             "content": "我叫什么名字?"}
        ],
        temperature=temp,
        max_tokens=300
    )
    print(f"===== temperature = {temp} =====")

    print(resp.choices[0].message.content)
#temperature是采样随机性参数，范围通常0-2。温度越高，每次回答越不一样、越发散；温度越低越稳定保守。

#4.2多轮对话——让模型“记住上下文

In [2]:
import os
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()

client = OpenAI(api_key=os.getenv("DEEPSEEK_API_KEY"), base_url="https://api.deepseek.com")

messages = [{
    "role": "system",
    "content": "你是一个简洁、友善的助手，回答不超过100字。"
}]

print("开始对话(输入exit退出):")
while True:
    user_input = input("你 > ")
    if user_input.strip().lower() == "exit":
        break
    messages.append({"role": "user", "content": user_input})

    resp = client.chat.completions.create(
        model="deepseek-v4-flash",
        messages=messages,
        temperature=0.5,
        max_tokens=500
    )
    reply = resp.choices[0].message.content
    print(f"AI > {reply}")

    messages.append({"role": "assistant", "content": reply})

开始对话(输入exit退出):
AI > 你好，李雷！很高兴认识你。有什么我可以帮你的吗？
AI > 你叫李雷呀，刚才你自己告诉我的。


#5.2结构化输出——让模型返回JSON

In [8]:
import json
import os
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()


#解析容错：模型偶尔会输出带 ```json 包裹或多余文字，直接 json.loads 会炸。写一个安全解析函数：
def safe_json_loads(text):
    """把模型输出安全解析成dict，容忍markdown代码块包裹"""
    text = text.strip()
    if text.startswith("'''"):
        text = text.strip("'").lstrip("json").strip()
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        #兜底：找出第一对{}再解析
        start = text.find("{")
        end = text.rfind("}")
        return json.loads(text[start:end+1])

client = OpenAI(api_key=os.getenv("DEEPSEEK_API_KEY"), base_url="https://api.deepseek.com")

prompt = """请把下面这段产品描述整理成JSON，字段必须包含：title(产品名),category(分类),price(价格,数字),features(卖点列表,至少3个)
产品描述:这是一款智能保温杯，可以显示水温、提醒喝水，续航30天，售价199元，适合上班族和学生。"""

resp = client.chat.completions.create(
    model="deepseek-v4-flash",
    messages=[{
        "role": "system",
        "content": "你只输出合法的JSON,不要输出任何其他文字,不要用markdown代码块包裹。"},
        {"role": "user",
         "content": prompt
         }],
    response_format={
        "type": "json_object"
    },
    temperature=0.2
)

raw = resp.choices[0].message.content
print("模型原始输出:\n", raw)

data = safe_json_loads(raw)  #解析成dict
print("解析后:\n", data)
print("产品名:", data["title"])
print("价格:", data["price"])

模型原始输出:
 {"title":"智能保温杯","category":"智能水杯","price":199,"features":["显示水温","提醒喝水","续航30天"]}
解析后:
 {'title': '智能保温杯', 'category': '智能水杯', 'price': 199, 'features': ['显示水温', '提醒喝水', '续航30天']}
产品名: 智能保温杯
价格: 199


#6.3双模型对比脚本

In [9]:
import os
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()

def ask(model,base_url,api_key,question):
    client = OpenAI(api_key=api_key,base_url=base_url)
    resp = client.chat.completions.create(
        model=model,
        messages=[{"role":"user","content":question}],
        max_tokens=300
    )
    return resp.choices[0].message.content,resp.usage.total_tokens

question = "用一段话解释什么是RAG,并举例说明它的应用场景。"

content1,tokens1 = ask("deepseek-v4-flash","https://api.deepseek.com",os.getenv("DEEPSEEK_API_KEY"),question)

content2,tokens2 = ask("qwen3.7-plus",os.getenv("DASHSCOPE_BASE_URL"),os.getenv("DASHSCOPE_API_KEY"),question)

print(f"【DeepSeek】消耗{tokens1}tokens\n{content1}\n")
print(f"【通义千问】消耗{tokens2}tokens\n{content2}\n")

【DeepSeek】消耗256tokens
RAG（Retrieval-Augmented Generation，检索增强生成）是一种结合信息检索与大语言模型的技术，它在模型生成回答前，先从外部知识库或文档中检索出相关片段，将其作为上下文输入给模型，从而让生成结果更准确、更新鲜且可溯源。例如，企业客服机器人可以接入产品手册和售后文档，当用户询问“某型号设备如何重置密码”时，系统先从文档中检索对应操作步骤，再让模型据此生成自然语言回复，既避免了模型“凭空编造”，又能即时反映最新政策或产品变动。

【通义千问】消耗800tokens
RAG（检索增强生成）是一种将信息检索与大语言模型相结合的技术，它让模型在生成回答前，先从外部知识库中检索出相关的实时或私有数据作为参考上下文，从而有效克服模型知识过时和“幻觉”问题，提升回答的准确性与专业性；例如在企业内部智能问答场景中，当员工询问“公司最新的差旅报销标准是什么”时，RAG系统会先从公司最新的内部财务制度文档中精准检索出相关条款，再将其作为背景知识输入给大模型，最终生成清晰、准确且符合公司现行规定的解答，而不是依赖模型可能错误或过时的预训练记忆。



#7.3小项目:简历优化助手

In [12]:
import json
import os
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()

client = OpenAI(api_key=os.getenv("DEEPSEEK_API_KEY"),base_url="https://api.deepseek.com")

def optimize_resume(raw_text,job_title):
    system = (
        "你是一位资深的技术面试官兼职业顾问。"
        "只输出合法JSON，不要输出 其他文字。"
    )
    user = f"""请优化以下简历条目，应聘岗位：{job_title}。
            原始内容：{raw_text}
            输出JSON格式，字段如下:
            -problems:数组，列出原条目的3个问题（空泛、缺量化、缺技术点等）
            -suggestion:字符串，优化的核心思路（100字内）
            -optimized:数组，优化后的简历条目（每条包含technique技术点，action动作，result可量化结果）
"""
    resp = client.chat.completions.create(
        model="deepseek-v4-flash",
        messages=[
            {
                "role":"system",
                "content":system},
            {
                "role":"user",
                "content":user}
        ],
        response_format={"type":"json_object"},
        temperature=0.3,
        max_tokens=800
    )
    return json.loads(resp.choices[0].message.content)

raw = "参与了智能问答系统的开发，负责一些模型相关的工作，协助完成了项目上线。"
result = optimize_resume(raw,"AI应用开发工程师")

print(json.dumps(result,ensure_ascii=False,indent=2))


{
  "problems": [
    "描述空泛，未说明具体技术栈和算法模型",
    "缺乏量化指标，无法体现性能和效果",
    "未体现个人职责和协作贡献"
  ],
  "suggestion": "突出具体技术（如BERT、RAG）、明确个人贡献、使用量化指标（准确率、响应时间、上线效果），并体现从开发到上线的完整闭环。",
  "optimized": [
    {
      "technique": "基于BERT的意图识别与实体抽取",
      "action": "负责智能问答系统核心模型选型与微调，优化语义理解准确率",
      "result": "意图识别准确率提升12%，实体抽取F1达到0.89"
    },
    {
      "technique": "检索增强生成（RAG）与知识库构建",
      "action": "设计并实现RAG流程，接入企业知识库，解决模型幻觉问题",
      "result": "回答相关性评分提升18%，幻觉率降低30%"
    },
    {
      "technique": "FastAPI服务部署与性能调优",
      "action": "主导模型服务化部署，设计接口并优化推理延迟",
      "result": "上线后平均响应时间从1.2s降至0.6s，支持500QPS，稳定运行3个月"
    }
  ]
}


#8.2流式调用

In [14]:
import os
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()
client = OpenAI(api_key=os.getenv("DEEPSEEK_API_KEY"),base_url="https://api.deepseek.com")

response = client.chat.completions.create(
    model="deepseek-v4-flash",
    messages=[{
        "role":"user",
        "content":"用300字介绍RAG检索增强生成技术"
    }],
    stream = True,
    max_tokens=500
)

for chunk in response:
    delta = chunk.choices[0].delta.content
    if delta:
        print(delta,end="",flush=True)

RAG（检索增强生成）是一种结合信息检索与大语言模型的AI技术。其核心思路是：在模型生成回答前，先从外部知识库（如文档、数据库、网页）中检索相关文本片段，将检索结果与用户问题一起输入大模型，引导模型基于事实内容生成答案。

相比传统纯生成式模型，RAG具有三大优势：  
1. **知识实时**：可动态更新外部知识库，无需重新训练模型即可获取最新信息。  
2. **减少幻觉**：生成过程引用检索证据，回答更准确、可验证。  
3. **领域适应**：企业可接入私有知识库，实现定制化问答、客服等应用。

典型流程包括：文档切分→向量化嵌入→存入向量数据库→用户查询时语义检索→拼接上下文→大模型生成。RAG已成为构建可靠AI助手的主流方案，广泛用于智能客服、知识管理和辅助写作等场景。